In [1]:
# ============================================================
# BLOCK 1: Environment Verification
# Check Python, PyTorch, and GPU availability
# ============================================================

import torch
import sys

# Print Python and PyTorch versions to confirm environment
print(f"Python version: {sys.version.split()[0]}")
print(f"PyTorch version: {torch.__version__}")

# Check if CUDA (GPU) is available - critical for fast inference
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    # Display GPU details: name and total memory in GB
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU memory: {gpu_mem:.1f} GB")
    print("✅ Ready to proceed!")
else:
    # If GPU is missing, training/inference will be 10-20x slower
    print("⚠️ No GPU detected!")
    print("   Go to: Runtime → Change runtime type → T4 GPU")

Python version: 3.12.13
PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
GPU memory: 15.6 GB
✅ Ready to proceed!


In [2]:
# ============================================================
# BLOCK 2: Install Required Libraries
# Run once per Colab session - takes ~2-3 minutes
# ============================================================

# Core ML & Transformers
!pip install -q transformers                    # HuggingFace models
!pip install -q sentence-transformers           # Embedding models
!pip install -q accelerate                      # Faster model loading

# Retrieval components for Advanced RAG
!pip install -q faiss-cpu                       # Vector similarity search
!pip install -q rank-bm25                       # Keyword-based search (BM25)

# Document processing
!pip install -q pymupdf                         # PDF text extraction
!pip install -q langchain langchain-community   # Text chunking utilities

# UI & Evaluation
!pip install -q gradio                          # Web interface
!pip install -q ragas datasets                  # RAG evaluation framework

print("✅ All libraries installed successfully!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 79.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 84.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 51.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.4/542.4 kB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━

In [3]:
# ============================================================
# BLOCK 3: Build Curated Medical Knowledge Base
# Replaces PDF download — embeds trusted content directly
# Content adapted from WHO, ADA, and CDC public guidelines
# ============================================================

import os

# Create directory for our knowledge base
KB_DIR = "knowledge_base"
os.makedirs(KB_DIR, exist_ok=True)

# Curated medical knowledge base
# Each entry is a self-contained medical reference document
medical_kb = {

    "diabetes_glucose.txt": """
DIABETES AND BLOOD GLUCOSE MANAGEMENT
Source: Adapted from ADA Standards of Care and WHO Diabetes Guidelines

NORMAL FASTING GLUCOSE RANGES
Normal fasting blood glucose is between 70 and 99 mg/dL (3.9 to 5.5 mmol/L).
Fasting glucose between 100 and 125 mg/dL indicates prediabetes.
Fasting glucose of 126 mg/dL or higher on two separate tests indicates diabetes mellitus.

POSTPRANDIAL GLUCOSE
Two-hour postprandial glucose under 140 mg/dL is considered normal.
Values between 140 and 199 mg/dL suggest impaired glucose tolerance.
A two-hour postprandial reading of 200 mg/dL or higher confirms diabetes diagnosis.

HYPERGLYCEMIA WARNING SIGNS
Blood glucose above 180 mg/dL is considered hyperglycemia.
Symptoms include increased thirst, frequent urination, blurred vision, and fatigue.
Sustained hyperglycemia above 250 mg/dL requires immediate clinical attention.

DIABETIC KETOACIDOSIS (DKA)
DKA is a life-threatening complication occurring when glucose exceeds 250 mg/dL combined with ketone production.
Warning signs include nausea, vomiting, abdominal pain, fruity breath odor, and rapid breathing.
This is a medical emergency requiring hospital treatment with IV fluids and insulin.

HYPOGLYCEMIA
Blood glucose below 70 mg/dL is hypoglycemia.
Symptoms include shakiness, sweating, confusion, irritability, and rapid heartbeat.
Severe hypoglycemia below 54 mg/dL can cause loss of consciousness and seizures.
Treatment: consume 15 grams of fast-acting carbohydrates and recheck in 15 minutes.

HEMOGLOBIN A1C TARGETS
A1C below 5.7 percent is normal.
A1C between 5.7 and 6.4 percent indicates prediabetes.
A1C of 6.5 percent or higher confirms diabetes.
For most adults with diabetes, the target A1C is below 7.0 percent.

GLUCOSE MONITORING RECOMMENDATIONS
Patients on insulin should check glucose at least four times daily.
Type 2 diabetes patients on oral medications should check fasting glucose daily.
Continuous glucose monitors are recommended for type 1 diabetes and insulin-treated type 2 diabetes.
""",

    "hypertension_bp.txt": """
HYPERTENSION AND BLOOD PRESSURE MANAGEMENT
Source: Adapted from JNC 8 Guidelines and WHO Hypertension Recommendations

BLOOD PRESSURE CATEGORIES
Normal blood pressure is below 120/80 mmHg.
Elevated blood pressure ranges from 120 to 129 systolic and below 80 diastolic.
Stage 1 hypertension is 130 to 139 systolic or 80 to 89 diastolic.
Stage 2 hypertension is 140 or higher systolic or 90 or higher diastolic.
Hypertensive crisis is systolic above 180 or diastolic above 120 mmHg.

HYPERTENSIVE EMERGENCY
Systolic blood pressure above 180 mmHg or diastolic above 120 mmHg with symptoms is a medical emergency.
Warning symptoms include severe headache, chest pain, shortness of breath, vision changes, and confusion.
Patients with these readings and symptoms should call emergency services immediately.

HYPERTENSIVE URGENCY
Severely elevated blood pressure without acute organ damage symptoms is hypertensive urgency.
Patients should contact their healthcare provider promptly for medication adjustment.
Avoid sudden large drops in blood pressure as this can cause harm.

HYPOTENSION
Blood pressure below 90/60 mmHg is generally considered hypotension.
Symptoms include dizziness, fainting, blurred vision, fatigue, and nausea.
Severe hypotension can indicate shock and requires emergency evaluation.

LIFESTYLE INTERVENTIONS FOR HYPERTENSION
The DASH diet emphasizes fruits, vegetables, whole grains, and low-fat dairy.
Sodium intake should be limited to less than 2300 mg per day, ideally 1500 mg.
Regular aerobic exercise of 150 minutes per week reduces blood pressure.
Weight loss of 1 kilogram reduces systolic blood pressure by approximately 1 mmHg.
Limiting alcohol to two drinks daily for men and one for women is recommended.

PHARMACOLOGICAL TREATMENT
First-line medications include thiazide diuretics, ACE inhibitors, ARBs, and calcium channel blockers.
For stage 2 hypertension, combination therapy with two medications is typically initiated.
Blood pressure targets are below 130/80 mmHg for most adults.
Diabetic and chronic kidney disease patients have stricter targets.

MONITORING RECOMMENDATIONS
Home blood pressure monitoring is encouraged with validated devices.
Take readings twice daily, morning and evening, for one week before clinic visits.
Average multiple readings rather than relying on a single measurement.
""",

    "cardiovascular.txt": """
CARDIOVASCULAR HEALTH AND HEART RATE
Source: Adapted from WHO HEARTS Technical Package and AHA Guidelines

NORMAL HEART RATE RANGES
Normal resting heart rate for adults is 60 to 100 beats per minute (bpm).
Trained athletes may have resting heart rates as low as 40 to 60 bpm.
Children have higher normal heart rates that decrease with age.

TACHYCARDIA
Heart rate above 100 bpm at rest is tachycardia.
Sinus tachycardia is often a normal response to exercise, stress, or fever.
Sustained tachycardia above 120 bpm at rest warrants medical evaluation.
Heart rate above 150 bpm with symptoms is potentially serious and may indicate arrhythmia.

BRADYCARDIA
Heart rate below 60 bpm in non-athletes is bradycardia.
Asymptomatic bradycardia in trained individuals is usually benign.
Heart rate below 50 bpm with symptoms such as dizziness, fatigue, or fainting requires evaluation.
Severe bradycardia below 40 bpm can indicate heart block and is a medical emergency.

ARRHYTHMIAS
Atrial fibrillation presents as irregular heart rhythm and increases stroke risk fivefold.
Symptoms include palpitations, fatigue, shortness of breath, and chest discomfort.
Patients with new-onset arrhythmia should undergo immediate ECG evaluation.

CHEST PAIN ASSESSMENT
Chest pain with shortness of breath, nausea, or arm pain may indicate myocardial infarction.
Symptoms lasting more than 20 minutes require emergency evaluation.
Risk factors include age over 50, diabetes, hypertension, smoking, and family history.
Calling emergency services and chewing aspirin can be life-saving while awaiting help.

CARDIOVASCULAR RISK FACTORS
Hypertension is the leading modifiable risk factor for cardiovascular disease.
Diabetes doubles the risk of heart disease and stroke.
LDL cholesterol above 100 mg/dL increases atherosclerosis risk.
Smoking, obesity, and physical inactivity significantly increase cardiovascular risk.

PREVENTION RECOMMENDATIONS
Aim for 150 minutes of moderate aerobic exercise per week.
Maintain BMI between 18.5 and 24.9 kg per square meter.
Mediterranean diet rich in fish, olive oil, and vegetables reduces cardiovascular events.
Annual cardiovascular risk assessment is recommended for adults over 40.
""",

    "emergency_thresholds.txt": """
MEDICAL EMERGENCY THRESHOLDS AND IMMEDIATE ACTIONS
Source: Adapted from WHO Emergency Care Guidelines

GLUCOSE EMERGENCIES
Blood glucose above 400 mg/dL with symptoms requires emergency evaluation.
Blood glucose below 54 mg/dL with confusion or unconsciousness requires immediate emergency response.
Hyperosmolar hyperglycemic state with glucose above 600 mg/dL is life-threatening.
Diabetic ketoacidosis with glucose above 250 mg/dL plus ketones requires hospital admission.

BLOOD PRESSURE EMERGENCIES
Systolic above 180 or diastolic above 120 with chest pain, shortness of breath, severe headache, vision changes, or neurological symptoms is a hypertensive emergency.
Systolic below 90 with symptoms of shock such as cold clammy skin, rapid pulse, and confusion requires emergency response.
Sudden severe blood pressure drop after standing with fainting episodes warrants immediate evaluation.

HEART RATE EMERGENCIES
Heart rate above 150 bpm at rest with chest pain, shortness of breath, or fainting requires emergency response.
Heart rate below 40 bpm with symptoms of decreased perfusion such as dizziness or chest pain is an emergency.
Irregular heart rhythm with chest pain or syncope requires immediate ECG evaluation.

WHEN TO CALL EMERGENCY SERVICES
Sudden chest pain or pressure lasting more than five minutes.
Difficulty breathing, severe shortness of breath, or inability to speak full sentences.
Sudden weakness or numbness on one side of the body suggesting stroke.
Loss of consciousness or severe altered mental status.
Severe bleeding that cannot be controlled with direct pressure.

FIRST AID FOR HYPOGLYCEMIA
For conscious patients, give 15 grams of fast-acting carbohydrates such as glucose tablets or fruit juice.
Recheck blood glucose after 15 minutes; repeat if still below 70 mg/dL.
For unconscious patients, do not give anything by mouth.
Administer glucagon injection if available and call emergency services.

FIRST AID FOR HYPERTENSIVE CRISIS
Have the patient sit down and remain calm.
Do not attempt to rapidly lower blood pressure with multiple medications.
Call emergency services if symptoms include chest pain, shortness of breath, or neurological changes.
Avoid giving sublingual nifedipine which can cause dangerous rapid drops.

GENERAL EMERGENCY PREPAREDNESS
Patients with chronic conditions should keep an updated medication list accessible.
Wear medical identification jewelry indicating conditions like diabetes or heart disease.
Family members should know warning signs and emergency contact procedures.
Home glucose meters and blood pressure monitors should be regularly calibrated.
"""
}

# Save each document to disk and load into memory
documents = {}  # Will be used by Block 5 (chunking)

for filename, content in medical_kb.items():
    # Save to file (so it persists like a real knowledge base)
    filepath = os.path.join(KB_DIR, filename)
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(content.strip())

    # Also load into memory for next block
    documents[filename] = content.strip()

    word_count = len(content.split())
    print(f"✓ Created: {filename} ({word_count} words)")

print(f"\n📊 Knowledge base ready:")
print(f"   Documents: {len(documents)}")
print(f"   Total words: {sum(len(d.split()) for d in documents.values()):,}")
print(f"   Location: ./{KB_DIR}/")

✓ Created: diabetes_glucose.txt (282 words)
✓ Created: hypertension_bp.txt (330 words)
✓ Created: cardiovascular.txt (315 words)
✓ Created: emergency_thresholds.txt (371 words)

📊 Knowledge base ready:
   Documents: 4
   Total words: 1,298
   Location: ./knowledge_base/


In [4]:
# ============================================================
# Install LangChain text splitter (new modular package)
# Run this once, then proceed to Block 4
# ============================================================

!pip install -q langchain-text-splitters

print("✅ langchain-text-splitters installed!")

✅ langchain-text-splitters installed!


In [5]:
# ============================================================
# BLOCK 4: Split Documents into Chunks
# Uses the new modular langchain-text-splitters package
# ============================================================

# New import path (langchain v0.2+)
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Configure the text splitter
# - chunk_size: target size of each chunk (in characters)
# - chunk_overlap: characters shared between consecutive chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,           # ~100 words per chunk - fits SLM context
    chunk_overlap=50,         # 10% overlap preserves context across chunks
    length_function=len,      # Measure length in characters
    separators=["\n\n", "\n", ". ", " ", ""]  # Split priority order
)

# Storage for all chunks across all documents
all_chunks = []           # List of chunk text strings
chunk_metadata = []       # Source info for each chunk (for citations)

# Process each document from Block 3
for filename, full_text in documents.items():
    chunks = text_splitter.split_text(full_text)

    for i, chunk in enumerate(chunks):
        all_chunks.append(chunk)
        chunk_metadata.append({
            "source": filename,
            "chunk_id": i,
            "chunk_size": len(chunk)
        })

    print(f"📑 {filename}: split into {len(chunks)} chunks")

# Final statistics
print(f"\n📊 Chunking summary:")
print(f"   Total chunks created: {len(all_chunks)}")
print(f"   Average chunk size: {sum(len(c) for c in all_chunks) / len(all_chunks):.0f} chars")

# Preview first chunk
print(f"\n🔍 Preview of first chunk:")
print(f"   Source: {chunk_metadata[0]['source']}")
print(f"   Content: {all_chunks[0][:200]}...")

📑 diabetes_glucose.txt: split into 6 chunks
📑 hypertension_bp.txt: split into 6 chunks
📑 cardiovascular.txt: split into 7 chunks
📑 emergency_thresholds.txt: split into 7 chunks

📊 Chunking summary:
   Total chunks created: 26
   Average chunk size: 351 chars

🔍 Preview of first chunk:
   Source: diabetes_glucose.txt
   Content: DIABETES AND BLOOD GLUCOSE MANAGEMENT
Source: Adapted from ADA Standards of Care and WHO Diabetes Guidelines

NORMAL FASTING GLUCOSE RANGES
Normal fasting blood glucose is between 70 and 99 mg/dL (3.9...


In [6]:
# ============================================================
# BLOCK 5: IoT Medical Sensor Simulation
# Simulates 3 medical devices producing realistic readings
# ============================================================

import random
from dataclasses import dataclass
from typing import Literal

# Set seed for reproducible results across runs
# Change to None for truly random readings each time
random.seed(42)


@dataclass
class IoTReading:
    """
    Represents a single set of readings from all simulated devices.
    Stores values + severity level + patient context.
    """
    glucose: float              # mg/dL — blood sugar
    bp_systolic: int            # mmHg — top blood pressure number
    bp_diastolic: int           # mmHg — bottom blood pressure number
    heart_rate: int             # bpm — pulse
    severity: str               # "normal" / "warning" / "critical"
    patient_age: int            # Patient age provides clinical context

    def __str__(self):
        """Format readings as a human-readable summary."""
        return (
            f"Glucose: {self.glucose} mg/dL | "
            f"BP: {self.bp_systolic}/{self.bp_diastolic} mmHg | "
            f"HR: {self.heart_rate} bpm | "
            f"Age: {self.patient_age} | "
            f"Severity: {self.severity.upper()}"
        )


def simulate_glucometer(severity: str) -> float:
    """
    Simulate a blood glucose reading.
    Returns value in mg/dL based on requested severity level.
    """
    if severity == "normal":
        # Normal fasting glucose: 70-99 mg/dL
        return round(random.uniform(70, 99), 1)
    elif severity == "warning":
        # Hyperglycemia / prediabetes range
        return round(random.uniform(100, 199), 1)
    elif severity == "critical":
        # Diabetic emergency range — DKA risk
        return round(random.uniform(200, 450), 1)


def simulate_blood_pressure(severity: str) -> tuple:
    """
    Simulate blood pressure reading.
    Returns (systolic, diastolic) in mmHg.
    """
    if severity == "normal":
        sys = random.randint(90, 119)
        dia = random.randint(60, 79)
    elif severity == "warning":
        # Stage 1 hypertension territory
        sys = random.randint(120, 159)
        dia = random.randint(80, 99)
    elif severity == "critical":
        # Hypertensive crisis — emergency
        sys = random.randint(160, 220)
        dia = random.randint(100, 130)
    return sys, dia


def simulate_heart_rate(severity: str) -> int:
    """
    Simulate heart rate reading in beats per minute.
    """
    if severity == "normal":
        # Normal resting heart rate
        return random.randint(60, 100)
    elif severity == "warning":
        # Mild bradycardia or tachycardia
        return random.choice([
            random.randint(50, 59),    # mild bradycardia
            random.randint(101, 130)   # mild tachycardia
        ])
    elif severity == "critical":
        # Severe bradycardia or tachycardia — emergency
        return random.choice([
            random.randint(30, 49),    # severe bradycardia
            random.randint(131, 180)   # severe tachycardia
        ])


def generate_iot_reading(
    severity: Literal["normal", "warning", "critical"] = "normal",
    patient_age: int = 50
) -> IoTReading:
    """
    Main simulation function — generates a complete reading set.

    Args:
        severity: One of "normal", "warning", "critical"
        patient_age: Patient age in years (affects clinical interpretation)

    Returns:
        IoTReading object with all sensor values
    """
    # Get readings from all 3 simulated devices
    glucose = simulate_glucometer(severity)
    bp_sys, bp_dia = simulate_blood_pressure(severity)
    heart_rate = simulate_heart_rate(severity)

    # Bundle into structured object
    return IoTReading(
        glucose=glucose,
        bp_systolic=bp_sys,
        bp_diastolic=bp_dia,
        heart_rate=heart_rate,
        severity=severity,
        patient_age=patient_age
    )


# ────────────────────────────────────────────────────────────
# Demo: generate one reading at each severity level
# ────────────────────────────────────────────────────────────
print("🩺 IoT Sensor Simulation Demo\n")
print("=" * 70)

for level in ["normal", "warning", "critical"]:
    reading = generate_iot_reading(severity=level, patient_age=55)
    print(f"\n[{level.upper()} CASE]")
    print(f"  {reading}")

print("\n" + "=" * 70)
print("✅ IoT simulator ready — can generate readings on demand")

🩺 IoT Sensor Simulation Demo


[NORMAL CASE]
  Glucose: 88.5 mg/dL | BP: 90/68 mmHg | HR: 75 bpm | Age: 55 | Severity: NORMAL

[WARNING CASE]
  Glucose: 122.1 mg/dL | BP: 126/97 mmHg | HR: 119 bpm | Age: 55 | Severity: WARNING

[CRITICAL CASE]
  Glucose: 207.9 mg/dL | BP: 165/106 mmHg | HR: 37 bpm | Age: 55 | Severity: CRITICAL

✅ IoT simulator ready — can generate readings on demand


In [7]:
# ============================================================
# BLOCK 6: Semantic Search with FAISS
# Builds vector index over medical chunks for meaning-based search
# ============================================================

import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

# ─────────────────────────────────────────────────────────────
# 1. Load the embedding model
# ─────────────────────────────────────────────────────────────
print("⬇ Loading embedding model (first time: ~30 seconds)...")

# all-MiniLM-L6-v2: small (80MB) but high-quality general-purpose embedder
# Outputs 384-dimensional vectors for any input text
embedder = SentenceTransformer("all-MiniLM-L6-v2")

# Move to GPU if available (10-20x faster encoding)
if torch.cuda.is_available():
    embedder = embedder.to("cuda")
    print("✓ Embedder loaded on GPU")
else:
    print("✓ Embedder loaded on CPU")


# ─────────────────────────────────────────────────────────────
# 2. Encode all chunks into vectors
# ─────────────────────────────────────────────────────────────
print(f"\n🔢 Encoding {len(all_chunks)} chunks into vectors...")

# encode() converts each text chunk to a 384-dim vector
# show_progress_bar=True gives visual feedback for large jobs
chunk_embeddings = embedder.encode(
    all_chunks,
    show_progress_bar=True,
    convert_to_numpy=True       # FAISS needs numpy arrays
)

# FAISS expects float32, not float64 (saves memory)
chunk_embeddings = chunk_embeddings.astype("float32")

print(f"✓ Embeddings shape: {chunk_embeddings.shape}")
print(f"  → {chunk_embeddings.shape[0]} chunks × {chunk_embeddings.shape[1]} dimensions")


# ─────────────────────────────────────────────────────────────
# 3. Build the FAISS index
# ─────────────────────────────────────────────────────────────
print("\n🗂️  Building FAISS index...")

# Get vector dimension from our embeddings (384 for MiniLM)
dim = chunk_embeddings.shape[1]

# IndexFlatL2: exact search using Euclidean (L2) distance
# Best for small-to-medium collections where 100% accuracy matters
faiss_index = faiss.IndexFlatL2(dim)

# Add all our chunk vectors to the index
faiss_index.add(chunk_embeddings)

print(f"✓ FAISS index built with {faiss_index.ntotal} vectors")


# ─────────────────────────────────────────────────────────────
# 4. Test the semantic search
# ─────────────────────────────────────────────────────────────
def semantic_search(query: str, top_k: int = 5):
    """
    Search the medical knowledge base by semantic meaning.

    Args:
        query: Natural language question
        top_k: Number of best-matching chunks to return

    Returns:
        List of (chunk_text, source, distance_score)
    """
    # Convert query to vector using the same embedder
    query_vector = embedder.encode([query], convert_to_numpy=True).astype("float32")

    # Search index — returns distances and indices of top_k closest chunks
    distances, indices = faiss_index.search(query_vector, top_k)

    # Build results list with chunk text + metadata
    results = []
    for idx, dist in zip(indices[0], distances[0]):
        results.append({
            "text": all_chunks[idx],
            "source": chunk_metadata[idx]["source"],
            "score": float(dist)            # lower = more similar
        })
    return results


# Test with a sample medical query
print("\n" + "=" * 70)
print("🔍 Testing semantic search...\n")

test_query = "What does very high blood sugar mean and what should I do?"
print(f"Query: {test_query}\n")

results = semantic_search(test_query, top_k=3)

for i, r in enumerate(results, 1):
    print(f"--- Result #{i} ---")
    print(f"Source: {r['source']}")
    print(f"Score:  {r['score']:.3f}  (lower = more relevant)")
    print(f"Text:   {r['text'][:200]}...")
    print()

print("=" * 70)
print("✅ Semantic search ready")

⬇ Loading embedding model (first time: ~30 seconds)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✓ Embedder loaded on GPU

🔢 Encoding 26 chunks into vectors...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

✓ Embeddings shape: (26, 384)
  → 26 chunks × 384 dimensions

🗂️  Building FAISS index...
✓ FAISS index built with 26 vectors

🔍 Testing semantic search...

Query: What does very high blood sugar mean and what should I do?

--- Result #1 ---
Source: diabetes_glucose.txt
Score:  0.791  (lower = more relevant)
Text:   POSTPRANDIAL GLUCOSE
Two-hour postprandial glucose under 140 mg/dL is considered normal.
Values between 140 and 199 mg/dL suggest impaired glucose tolerance.
A two-hour postprandial reading of 200 mg/...

--- Result #2 ---
Source: diabetes_glucose.txt
Score:  0.877  (lower = more relevant)
Text:   HYPOGLYCEMIA
Blood glucose below 70 mg/dL is hypoglycemia.
Symptoms include shakiness, sweating, confusion, irritability, and rapid heartbeat.
Severe hypoglycemia below 54 mg/dL can cause loss of cons...

--- Result #3 ---
Source: emergency_thresholds.txt
Score:  1.027  (lower = more relevant)
Text:   MEDICAL EMERGENCY THRESHOLDS AND IMMEDIATE ACTIONS
Source: Adapted from WHO Emer

In [8]:
# ============================================================
# BLOCK 7: Keyword Search with BM25
# Complements semantic search by handling exact terms & numbers
# ============================================================

from rank_bm25 import BM25Okapi
import re


def tokenize(text: str) -> list:
    """
    Simple tokenizer for BM25.
    Lowercases and splits on non-alphanumeric characters.
    """
    # Lowercase and replace non-word chars with spaces
    text = text.lower()
    # \w = letters/digits/underscore
    tokens = re.findall(r"\w+", text)
    return tokens


# ─────────────────────────────────────────────────────────────
# 1. Tokenize all chunks
# ─────────────────────────────────────────────────────────────
print("🔤 Tokenizing all chunks for BM25...")

tokenized_chunks = [tokenize(chunk) for chunk in all_chunks]

# Show example tokenization
print(f"\nExample — first 10 tokens of chunk 0:")
print(f"  {tokenized_chunks[0][:10]}")


# ─────────────────────────────────────────────────────────────
# 2. Build the BM25 index
# ─────────────────────────────────────────────────────────────
print("\n🗂️  Building BM25 index...")

# BM25Okapi is the standard implementation of BM25
# It pre-computes term frequencies for fast querying
bm25 = BM25Okapi(tokenized_chunks)

print(f"✓ BM25 index built over {len(tokenized_chunks)} chunks")


# ─────────────────────────────────────────────────────────────
# 3. BM25 search function
# ─────────────────────────────────────────────────────────────
def keyword_search(query: str, top_k: int = 5):
    """
    Search using BM25 keyword matching.

    Args:
        query: Natural language query
        top_k: Number of top chunks to return

    Returns:
        List of dicts with text, source, and BM25 score
    """
    # Tokenize the query the same way as the chunks
    query_tokens = tokenize(query)

    # Get BM25 scores for ALL chunks against this query
    scores = bm25.get_scores(query_tokens)

    # Get indices of top_k highest scores (sorted descending)
    top_indices = np.argsort(scores)[::-1][:top_k]

    # Build results
    results = []
    for idx in top_indices:
        results.append({
            "text": all_chunks[idx],
            "source": chunk_metadata[idx]["source"],
            "score": float(scores[idx])     # higher = more relevant
        })
    return results


# ─────────────────────────────────────────────────────────────
# 4. Test BM25 search
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("🔍 Testing BM25 keyword search...\n")

test_query = "diabetic ketoacidosis DKA emergency"
print(f"Query: {test_query}\n")

results = keyword_search(test_query, top_k=3)

for i, r in enumerate(results, 1):
    print(f"--- Result #{i} ---")
    print(f"Source: {r['source']}")
    print(f"Score:  {r['score']:.3f}  (higher = more relevant)")
    print(f"Text:   {r['text'][:200]}...")
    print()

print("=" * 70)
print("✅ BM25 keyword search ready")

🔤 Tokenizing all chunks for BM25...

Example — first 10 tokens of chunk 0:
  ['diabetes', 'and', 'blood', 'glucose', 'management', 'source', 'adapted', 'from', 'ada', 'standards']

🗂️  Building BM25 index...
✓ BM25 index built over 26 chunks

🔍 Testing BM25 keyword search...

Query: diabetic ketoacidosis DKA emergency

--- Result #1 ---
Source: diabetes_glucose.txt
Score:  8.748  (higher = more relevant)
Text:   DIABETIC KETOACIDOSIS (DKA)
DKA is a life-threatening complication occurring when glucose exceeds 250 mg/dL combined with ketone production.
Warning signs include nausea, vomiting, abdominal pain, fru...

--- Result #2 ---
Source: emergency_thresholds.txt
Score:  4.002  (higher = more relevant)
Text:   MEDICAL EMERGENCY THRESHOLDS AND IMMEDIATE ACTIONS
Source: Adapted from WHO Emergency Care Guidelines

GLUCOSE EMERGENCIES
Blood glucose above 400 mg/dL with symptoms requires emergency evaluation.
Bl...

--- Result #3 ---
Source: hypertension_bp.txt
Score:  1.983  (higher = more

In [9]:
# ============================================================
# BLOCK 8: Hybrid Search using Reciprocal Rank Fusion (RRF)
# Combines FAISS semantic + BM25 keyword results
# ============================================================

def hybrid_search(query: str, top_k: int = 5, fetch_k: int = 10, rrf_k: int = 60):
    """
    Hybrid search using Reciprocal Rank Fusion (RRF).

    Strategy:
    1. Get top fetch_k results from BOTH semantic and keyword search
    2. Score each chunk by RRF: sum of 1/(rrf_k + rank) across systems
    3. Return top_k chunks ranked by combined RRF score

    Args:
        query: User query
        top_k: Final number of chunks to return
        fetch_k: How many to retrieve from each system (more = better fusion)
        rrf_k: RRF constant — 60 is the standard from Cormack et al. (2009)

    Returns:
        List of dicts with chunk text, source, and combined score
    """

    # ─────────────────────────────────────────────────────────
    # 1. Get results from both retrieval systems
    # ─────────────────────────────────────────────────────────
    semantic_results = semantic_search(query, top_k=fetch_k)
    keyword_results = keyword_search(query, top_k=fetch_k)

    # ─────────────────────────────────────────────────────────
    # 2. Build a dict to accumulate RRF scores
    #    Key = chunk text (acts as unique identifier)
    # ─────────────────────────────────────────────────────────
    rrf_scores = {}     # chunk_text → combined RRF score
    chunk_info = {}     # chunk_text → {source, text} (for output)

    # Score each chunk from semantic search
    # rank starts at 1 (not 0) for the RRF formula
    for rank, result in enumerate(semantic_results, start=1):
        chunk = result["text"]
        # RRF formula: 1 / (k + rank)
        rrf_scores[chunk] = rrf_scores.get(chunk, 0) + 1 / (rrf_k + rank)
        chunk_info[chunk] = {
            "text": result["text"],
            "source": result["source"]
        }

    # Score each chunk from keyword search
    # If a chunk appears in both, scores ADD together (boost!)
    for rank, result in enumerate(keyword_results, start=1):
        chunk = result["text"]
        rrf_scores[chunk] = rrf_scores.get(chunk, 0) + 1 / (rrf_k + rank)
        chunk_info[chunk] = {
            "text": result["text"],
            "source": result["source"]
        }

    # ─────────────────────────────────────────────────────────
    # 3. Sort by RRF score (descending) and return top_k
    # ─────────────────────────────────────────────────────────
    sorted_chunks = sorted(
        rrf_scores.items(),
        key=lambda x: x[1],
        reverse=True   # highest score first
    )[:top_k]

    # Build final results list
    results = []
    for chunk_text, rrf_score in sorted_chunks:
        results.append({
            "text": chunk_info[chunk_text]["text"],
            "source": chunk_info[chunk_text]["source"],
            "score": rrf_score
        })

    return results


# ─────────────────────────────────────────────────────────────
# Test the hybrid search
# ─────────────────────────────────────────────────────────────
print("=" * 70)
print("🔍 Testing Hybrid Search (FAISS + BM25 via RRF)\n")

# Same query that previously gave mixed results
test_query = "What does very high blood sugar mean and what should I do?"
print(f"Query: {test_query}\n")

results = hybrid_search(test_query, top_k=3)

for i, r in enumerate(results, 1):
    print(f"--- Result #{i} ---")
    print(f"Source:    {r['source']}")
    print(f"RRF Score: {r['score']:.5f}  (higher = more relevant)")
    print(f"Text:      {r['text'][:250]}...")
    print()

print("=" * 70)
print("✅ Hybrid search ready")

🔍 Testing Hybrid Search (FAISS + BM25 via RRF)

Query: What does very high blood sugar mean and what should I do?

--- Result #1 ---
Source:    emergency_thresholds.txt
RRF Score: 0.03126  (higher = more relevant)
Text:      FIRST AID FOR HYPOGLYCEMIA
For conscious patients, give 15 grams of fast-acting carbohydrates such as glucose tablets or fruit juice.
Recheck blood glucose after 15 minutes; repeat if still below 70 mg/dL.
For unconscious patients, do not give anythi...

--- Result #2 ---
Source:    hypertension_bp.txt
RRF Score: 0.03031  (higher = more relevant)
Text:      HYPERTENSIVE URGENCY
Severely elevated blood pressure without acute organ damage symptoms is hypertensive urgency.
Patients should contact their healthcare provider promptly for medication adjustment.
Avoid sudden large drops in blood pressure as thi...

--- Result #3 ---
Source:    diabetes_glucose.txt
RRF Score: 0.02991  (higher = more relevant)
Text:      DIABETES AND BLOOD GLUCOSE MANAGEMENT
Source: Adapted 

In [10]:
# ============================================================
# BLOCK 9: Cross-Encoder Re-Ranker
# Refines hybrid search results with deeper relevance scoring
# ============================================================

from sentence_transformers import CrossEncoder

# ─────────────────────────────────────────────────────────────
# 1. Load the cross-encoder model
# ─────────────────────────────────────────────────────────────
print("⬇ Loading cross-encoder re-ranker (~80MB, first time: ~30s)...")

# This model scores how relevant a (query, chunk) pair is on a single number
# Higher score = more relevant. Range varies but typically [-10, +10]
reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2",
    max_length=512,                                 # Truncate inputs longer than this
    device="cuda" if torch.cuda.is_available() else "cpu"
)

print("✓ Re-ranker ready")


# ─────────────────────────────────────────────────────────────
# 2. Re-ranking function
# ─────────────────────────────────────────────────────────────
def rerank(query: str, candidates: list, top_k: int = 3):
    """
    Re-rank candidate chunks using cross-encoder.

    Args:
        query: The user query
        candidates: List of dicts from hybrid_search() (each has 'text', 'source')
        top_k: How many top chunks to return after re-ranking

    Returns:
        Sorted list with re-ranked scores added
    """
    if not candidates:
        return []

    # Build (query, chunk) pairs — what cross-encoder expects
    pairs = [(query, c["text"]) for c in candidates]

    # Get relevance score for each pair
    # show_progress_bar=False to avoid clutter for small batches
    scores = reranker.predict(pairs, show_progress_bar=False)

    # Attach new scores to candidates
    for candidate, score in zip(candidates, scores):
        candidate["rerank_score"] = float(score)

    # Sort by rerank score (descending) and keep top_k
    reranked = sorted(
        candidates,
        key=lambda x: x["rerank_score"],
        reverse=True
    )[:top_k]

    return reranked


# ─────────────────────────────────────────────────────────────
# 3. Combined Advanced Retrieval Pipeline
# ─────────────────────────────────────────────────────────────
def advanced_retrieve(query: str, top_k: int = 3, hybrid_k: int = 10):
    """
    Full advanced retrieval pipeline:
    Hybrid Search (10 candidates) → Cross-Encoder Re-Rank → Top 3

    This is the complete retrieval part of Advanced RAG.
    Query rewriting (Block 10) and SLM generation (Block 11) come next.
    """
    # Step 1: Get a wider candidate pool from hybrid search
    candidates = hybrid_search(query, top_k=hybrid_k)

    # Step 2: Re-rank with cross-encoder for higher accuracy
    final_results = rerank(query, candidates, top_k=top_k)

    return final_results


# ─────────────────────────────────────────────────────────────
# 4. Test the full pipeline
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("🔍 Testing Full Advanced Retrieval (Hybrid → Re-Rank)\n")

test_query = "Patient has glucose 400 mg/dL and chest pain. What's the priority?"
print(f"Query: {test_query}\n")

results = advanced_retrieve(test_query, top_k=3)

for i, r in enumerate(results, 1):
    print(f"--- Result #{i} ---")
    print(f"Source:        {r['source']}")
    print(f"RRF Score:     {r['score']:.5f}")
    print(f"Rerank Score:  {r['rerank_score']:.3f}  ⭐ (final ranking)")
    print(f"Text:          {r['text'][:250]}...")
    print()

print("=" * 70)
print("✅ Advanced retrieval pipeline ready")
print("   Hybrid Search + Cross-Encoder Re-Ranking complete")

⬇ Loading cross-encoder re-ranker (~80MB, first time: ~30s)...


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

✓ Re-ranker ready

🔍 Testing Full Advanced Retrieval (Hybrid → Re-Rank)

Query: Patient has glucose 400 mg/dL and chest pain. What's the priority?

--- Result #1 ---
Source:        emergency_thresholds.txt
RRF Score:     0.03279
Rerank Score:  1.750  ⭐ (final ranking)
Text:          MEDICAL EMERGENCY THRESHOLDS AND IMMEDIATE ACTIONS
Source: Adapted from WHO Emergency Care Guidelines

GLUCOSE EMERGENCIES
Blood glucose above 400 mg/dL with symptoms requires emergency evaluation.
Blood glucose below 54 mg/dL with confusion or uncon...

--- Result #2 ---
Source:        diabetes_glucose.txt
RRF Score:     0.01515
Rerank Score:  -0.537  ⭐ (final ranking)
Text:          DIABETIC KETOACIDOSIS (DKA)
DKA is a life-threatening complication occurring when glucose exceeds 250 mg/dL combined with ketone production.
Warning signs include nausea, vomiting, abdominal pain, fruity breath odor, and rapid breathing.
This is a me...

--- Result #3 ---
Source:        diabetes_glucose.txt
RRF Score:     0.03

In [11]:
# ============================================================
# BLOCK 10: Query Rewriting (IoT readings → clinical question)
# Converts raw sensor numbers into a medically-formatted query
# ============================================================

def classify_glucose(value: float) -> dict:
    """
    Classify a glucose reading and return its medical description.
    """
    if value < 54:
        return {"label": "severe hypoglycemia", "concern": "critical"}
    elif value < 70:
        return {"label": "hypoglycemia", "concern": "warning"}
    elif value <= 99:
        return {"label": "normal fasting glucose", "concern": "normal"}
    elif value <= 125:
        return {"label": "impaired fasting glucose (prediabetes range)", "concern": "warning"}
    elif value <= 199:
        return {"label": "elevated glucose (possible diabetes range)", "concern": "warning"}
    elif value <= 250:
        return {"label": "hyperglycemia", "concern": "warning"}
    elif value <= 400:
        return {"label": "severe hyperglycemia", "concern": "critical"}
    else:
        # Above 400 mg/dL — DKA or HHS risk
        return {"label": "extreme hyperglycemia (DKA/HHS risk)", "concern": "critical"}


def classify_blood_pressure(systolic: int, diastolic: int) -> dict:
    """
    Classify blood pressure based on systolic/diastolic values.
    Uses JNC 8 / AHA categories.
    """
    if systolic < 90 or diastolic < 60:
        return {"label": "hypotension", "concern": "warning"}
    elif systolic < 120 and diastolic < 80:
        return {"label": "normal blood pressure", "concern": "normal"}
    elif systolic < 130 and diastolic < 80:
        return {"label": "elevated blood pressure", "concern": "warning"}
    elif systolic < 140 or diastolic < 90:
        return {"label": "stage 1 hypertension", "concern": "warning"}
    elif systolic < 180 and diastolic < 120:
        return {"label": "stage 2 hypertension", "concern": "warning"}
    else:
        # Hypertensive crisis territory
        return {"label": "hypertensive crisis", "concern": "critical"}


def classify_heart_rate(value: int) -> dict:
    """
    Classify heart rate as normal, bradycardia, or tachycardia.
    """
    if value < 40:
        return {"label": "severe bradycardia", "concern": "critical"}
    elif value < 60:
        return {"label": "bradycardia", "concern": "warning"}
    elif value <= 100:
        return {"label": "normal heart rate", "concern": "normal"}
    elif value <= 130:
        return {"label": "mild tachycardia", "concern": "warning"}
    else:
        # Above 130 bpm at rest = serious
        return {"label": "severe tachycardia", "concern": "critical"}


def rewrite_query(reading) -> dict:
    """
    Rewrite raw IoT readings into a structured clinical question.

    Args:
        reading: IoTReading object from Block 5

    Returns:
        Dict with:
        - 'query': natural-language clinical question for the RAG
        - 'classifications': per-measurement medical labels (for UI display)
        - 'overall_concern': highest severity across all readings
    """
    # ─────────────────────────────────────────────────────────
    # 1. Classify each reading
    # ─────────────────────────────────────────────────────────
    glucose_class = classify_glucose(reading.glucose)
    bp_class = classify_blood_pressure(reading.bp_systolic, reading.bp_diastolic)
    hr_class = classify_heart_rate(reading.heart_rate)

    # ─────────────────────────────────────────────────────────
    # 2. Determine overall severity
    #    (the worst of the three rules the case)
    # ─────────────────────────────────────────────────────────
    severity_order = {"normal": 0, "warning": 1, "critical": 2}
    all_concerns = [glucose_class["concern"], bp_class["concern"], hr_class["concern"]]
    overall = max(all_concerns, key=lambda x: severity_order[x])

    # ─────────────────────────────────────────────────────────
    # 3. Build a natural clinical question
    # ─────────────────────────────────────────────────────────
    # Start with patient context
    query_parts = [
        f"A {reading.patient_age}-year-old patient presents with the following readings:"
    ]

    # Add each measurement with its medical label
    query_parts.append(
        f"blood glucose of {reading.glucose} mg/dL ({glucose_class['label']}),"
    )
    query_parts.append(
        f"blood pressure of {reading.bp_systolic}/{reading.bp_diastolic} mmHg ({bp_class['label']}),"
    )
    query_parts.append(
        f"and heart rate of {reading.heart_rate} bpm ({hr_class['label']})."
    )

    # Closing question — adapts based on severity
    if overall == "critical":
        query_parts.append(
            "What are the immediate clinical concerns and emergency actions required?"
        )
    elif overall == "warning":
        query_parts.append(
            "What clinical conditions should be considered and what monitoring is recommended?"
        )
    else:
        query_parts.append(
            "Are these readings within normal range and what general health guidance applies?"
        )

    # Combine into one query string
    full_query = " ".join(query_parts)

    return {
        "query": full_query,
        "classifications": {
            "glucose": glucose_class,
            "blood_pressure": bp_class,
            "heart_rate": hr_class
        },
        "overall_concern": overall
    }


# ─────────────────────────────────────────────────────────────
# Test query rewriting on all 3 severity levels
# ─────────────────────────────────────────────────────────────
print("=" * 70)
print("✏️  Testing Query Rewriting\n")

for level in ["normal", "warning", "critical"]:
    # Generate IoT reading for this level
    reading = generate_iot_reading(severity=level, patient_age=58)

    # Rewrite into clinical query
    rewritten = rewrite_query(reading)

    print(f"\n{'─' * 70}")
    print(f"[CASE: {level.upper()}]")
    print(f"\n🩺 Raw IoT readings:")
    print(f"   {reading}")

    print(f"\n📋 Medical classifications:")
    for measure, info in rewritten["classifications"].items():
        emoji = "🟢" if info["concern"] == "normal" else "🟡" if info["concern"] == "warning" else "🔴"
        print(f"   {emoji} {measure:>15}: {info['label']}")

    print(f"\n⚠️  Overall concern: {rewritten['overall_concern'].upper()}")

    print(f"\n💬 Rewritten clinical query:")
    print(f"   \"{rewritten['query']}\"")

print("\n" + "=" * 70)
print("✅ Query rewriter ready")

✏️  Testing Query Rewriting


──────────────────────────────────────────────────────────────────────
[CASE: NORMAL]

🩺 Raw IoT readings:
   Glucose: 86.3 mg/dL | BP: 112/77 mmHg | HR: 86 bpm | Age: 58 | Severity: NORMAL

📋 Medical classifications:
   🟢         glucose: normal fasting glucose
   🟢  blood_pressure: normal blood pressure
   🟢      heart_rate: normal heart rate

⚠️  Overall concern: NORMAL

💬 Rewritten clinical query:
   "A 58-year-old patient presents with the following readings: blood glucose of 86.3 mg/dL (normal fasting glucose), blood pressure of 112/77 mmHg (normal blood pressure), and heart rate of 86 bpm (normal heart rate). Are these readings within normal range and what general health guidance applies?"

──────────────────────────────────────────────────────────────────────
[CASE: WARNING]

🩺 Raw IoT readings:
   Glucose: 121.8 mg/dL | BP: 157/88 mmHg | HR: 50 bpm | Age: 58 | Severity: WARNING

📋 Medical classifications:
   🟡         glucose: impaired fasting glu

In [12]:
# ============================================================
# Quick test: full retrieval pipeline (no SLM yet)
# ============================================================

# Generate a critical case
reading = generate_iot_reading(severity="critical", patient_age=62)
print(f"🩺 Reading: {reading}\n")

# Rewrite to clinical query
rewritten = rewrite_query(reading)
print(f"💬 Query: {rewritten['query']}\n")

# Run full advanced retrieval
chunks = advanced_retrieve(rewritten['query'], top_k=3)

print("📚 Top 3 retrieved chunks:")
for i, c in enumerate(chunks, 1):
    print(f"\n#{i} — {c['source']} (rerank: {c['rerank_score']:.2f})")
    print(f"   {c['text'][:150]}...")

🩺 Reading: Glucose: 225.6 mg/dL | BP: 184/103 mmHg | HR: 153 bpm | Age: 62 | Severity: CRITICAL

💬 Query: A 62-year-old patient presents with the following readings: blood glucose of 225.6 mg/dL (hyperglycemia), blood pressure of 184/103 mmHg (hypertensive crisis), and heart rate of 153 bpm (severe tachycardia). What are the immediate clinical concerns and emergency actions required?

📚 Top 3 retrieved chunks:

#1 — diabetes_glucose.txt (rerank: -0.00)
   POSTPRANDIAL GLUCOSE
Two-hour postprandial glucose under 140 mg/dL is considered normal.
Values between 140 and 199 mg/dL suggest impaired glucose tol...

#2 — diabetes_glucose.txt (rerank: -1.34)
   HYPOGLYCEMIA
Blood glucose below 70 mg/dL is hypoglycemia.
Symptoms include shakiness, sweating, confusion, irritability, and rapid heartbeat.
Severe ...

#3 — emergency_thresholds.txt (rerank: -1.36)
   MEDICAL EMERGENCY THRESHOLDS AND IMMEDIATE ACTIONS
Source: Adapted from WHO Emergency Care Guidelines

GLUCOSE EMERGENCIES
Blood glucose

In [13]:
# ============================================================
# BLOCK 11: SLM Response Generation
# Generate clinical explanations using Qwen2.5-1.5B-Instruct
# grounded in retrieved medical context
# ============================================================

from transformers import AutoTokenizer, AutoModelForCausalLM

# ─────────────────────────────────────────────────────────────
# 1. Load the SLM (one-time setup, ~2-3 minutes first run)
# ─────────────────────────────────────────────────────────────
print("⬇ Loading SLM (Qwen2.5-1.5B-Instruct, ~3GB)...")
print("   First run takes 2-3 minutes to download. Subsequent runs are fast.\n")

SLM_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

# Tokenizer: converts text ↔ token IDs that the model processes
tokenizer = AutoTokenizer.from_pretrained(SLM_NAME)

# Model: load with float16 to fit on T4 GPU (saves ~50% memory)
slm_model = AutoModelForCausalLM.from_pretrained(
    SLM_NAME,
    torch_dtype=torch.float16,    # half-precision, fits on Colab T4
    device_map="auto"             # auto-select GPU if available
)

# Switch to evaluation mode (disables dropout, etc.)
slm_model.eval()

print(f"✓ SLM loaded on: {slm_model.device}")
print(f"  Total parameters: {slm_model.num_parameters() / 1e9:.2f}B")


# ─────────────────────────────────────────────────────────────
# 2. Build the prompt with retrieved context
# ─────────────────────────────────────────────────────────────
def build_prompt(query: str, chunks: list) -> list:
    """
    Construct chat messages for the SLM, including retrieved context.

    Args:
        query: Rewritten clinical query (from Block 10)
        chunks: Top retrieved chunks (from Block 9)

    Returns:
        Chat messages in OpenAI-style format (system + user)
    """
    # Combine all chunks into a single context block, with source labels
    context_block = "\n\n".join([
        f"[Source: {c['source']}]\n{c['text']}"
        for c in chunks
    ])

    # System message: defines the model's role and constraints
    system_msg = (
        "You are a medical assistant that explains clinical readings to patients "
        "based on trusted medical guidelines. Always ground your answers in the "
        "provided context. Be clear, concise, and avoid speculation. If the "
        "readings indicate an emergency, clearly recommend seeking immediate care."
    )

    # User message: includes both the context and the question
    user_msg = (
        f"CLINICAL CONTEXT FROM MEDICAL GUIDELINES:\n"
        f"{context_block}\n\n"
        f"PATIENT QUERY:\n{query}\n\n"
        f"Provide a clear medical explanation suitable for the patient, "
        f"covering: (1) what the readings mean, (2) the level of concern, "
        f"and (3) recommended actions."
    )

    return [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": user_msg}
    ]


# ─────────────────────────────────────────────────────────────
# 3. Generate response from SLM
# ─────────────────────────────────────────────────────────────
def generate_response(messages: list, max_new_tokens: int = 400) -> str:
    """
    Run the SLM to generate a response from chat messages.
    """
    # Apply Qwen's chat template (adds special tokens like <|im_start|>)
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True   # adds the assistant turn marker
    )

    # Tokenize and move to GPU
    inputs = tokenizer(prompt, return_tensors="pt").to(slm_model.device)

    # Generate (no_grad = no gradient computation = faster + less memory)
    with torch.no_grad():
        output_ids = slm_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.3,             # low = focused, deterministic
            do_sample=True,
            top_p=0.9,                   # nucleus sampling
            repetition_penalty=1.1,      # mild penalty to prevent loops
            pad_token_id=tokenizer.eos_token_id
        )

    # Decode only the NEW tokens (skip the input prompt)
    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True)

    return response.strip()


# ─────────────────────────────────────────────────────────────
# 4. Full end-to-end pipeline function
# ─────────────────────────────────────────────────────────────
def run_full_pipeline(reading) -> dict:
    """
    Complete Advanced RAG pipeline:
    IoT Reading → Query Rewriting → Hybrid Retrieval → Re-Rank → SLM → Response
    """
    # Step 1: Rewrite IoT readings as clinical query
    rewritten = rewrite_query(reading)

    # Step 2: Retrieve top 3 most relevant chunks
    chunks = advanced_retrieve(rewritten["query"], top_k=3)

    # Step 3: Build prompt with retrieved context
    messages = build_prompt(rewritten["query"], chunks)

    # Step 4: Generate medical explanation
    response = generate_response(messages)

    # Return everything for display + evaluation
    return {
        "reading": reading,
        "query": rewritten["query"],
        "classifications": rewritten["classifications"],
        "overall_concern": rewritten["overall_concern"],
        "retrieved_chunks": chunks,
        "sources": list(set(c["source"] for c in chunks)),
        "response": response
    }


# ─────────────────────────────────────────────────────────────
# 5. Test the full pipeline on all severity levels
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("🩺 Testing Full Advanced RAG Pipeline\n")

# Generate a critical case
test_reading = generate_iot_reading(severity="critical", patient_age=58)

print(f"📊 PATIENT READINGS:")
print(f"   {test_reading}")

print(f"\n🔄 Running pipeline (IoT → Rewrite → Retrieve → Re-rank → SLM)...")

result = run_full_pipeline(test_reading)

print(f"\n📋 MEDICAL CLASSIFICATIONS:")
for measure, info in result["classifications"].items():
    emoji = "🟢" if info["concern"] == "normal" else "🟡" if info["concern"] == "warning" else "🔴"
    print(f"   {emoji} {measure}: {info['label']}")

print(f"\n⚠️  OVERALL CONCERN: {result['overall_concern'].upper()}")

print(f"\n💬 GENERATED MEDICAL EXPLANATION:")
print("─" * 70)
print(result["response"])
print("─" * 70)

print(f"\n📚 SOURCES USED:")
for s in result["sources"]:
    print(f"   • {s}")

print("\n" + "=" * 70)
print("✅ Full pipeline working end-to-end!")

⬇ Loading SLM (Qwen2.5-1.5B-Instruct, ~3GB)...
   First run takes 2-3 minutes to download. Subsequent runs are fast.



config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

✓ SLM loaded on: cuda:0
  Total parameters: 1.54B

🩺 Testing Full Advanced RAG Pipeline

📊 PATIENT READINGS:
   Glucose: 401.8 mg/dL | BP: 206/114 mmHg | HR: 138 bpm | Age: 58 | Severity: CRITICAL

🔄 Running pipeline (IoT → Rewrite → Retrieve → Re-rank → SLM)...

📋 MEDICAL CLASSIFICATIONS:
   🔴 glucose: extreme hyperglycemia (DKA/HHS risk)
   🔴 blood_pressure: hypertensive crisis
   🔴 heart_rate: severe tachycardia

⚠️  OVERALL CONCERN: CRITICAL

💬 GENERATED MEDICAL EXPLANATION:
──────────────────────────────────────────────────────────────────────
The patient's readings indicate significant health risks:

1. **Blood Glucose Level**: The blood glucose reading of 401.8 mg/dL is extremely high, indicating diabetic ketoacidosis (DKA) or hyperosmolar hyperglycemic state (HHS). This is a life-threatening condition where blood sugar levels are dangerously elevated, often associated with dehydration and electrolyte imbalances.

2. **Blood Pressure**: The systolic blood pressure of 206 mmHg an

In [16]:
# ============================================================
# BLOCK 12: Gradio UI for the Advanced RAG System
# Interactive interface for testing the pipeline
# ============================================================

import gradio as gr


def analyze(glucose, bp_sys, bp_dia, heart_rate, patient_age):
    """
    Main UI handler — runs the full pipeline and formats output.
    Called when the user clicks the 'Analyze' button.
    """
    # ─────────────────────────────────────────────────────────
    # 1. Build an IoTReading object from the slider values
    # ─────────────────────────────────────────────────────────
    # Determine severity ourselves (since user picked manual values)
    # This is a quick heuristic — full classification happens in Block 10
    if glucose > 250 or bp_sys > 180 or heart_rate > 130 or heart_rate < 50:
        severity = "critical"
    elif glucose > 140 or bp_sys > 130 or heart_rate > 100:
        severity = "warning"
    else:
        severity = "normal"

    reading = IoTReading(
        glucose=glucose,
        bp_systolic=int(bp_sys),
        bp_diastolic=int(bp_dia),
        heart_rate=int(heart_rate),
        severity=severity,
        patient_age=int(patient_age)
    )

    # ─────────────────────────────────────────────────────────
    # 2. Run the full Advanced RAG pipeline
    # ─────────────────────────────────────────────────────────
    result = run_full_pipeline(reading)

    # ─────────────────────────────────────────────────────────
    # 3. Format the classifications as a readable Markdown block
    # ─────────────────────────────────────────────────────────
    classifications_md = "### 📋 Medical Classifications\n\n"
    for measure, info in result["classifications"].items():
        # Map concern level to colored emoji
        emoji = {
            "normal": "🟢",
            "warning": "🟡",
            "critical": "🔴"
        }[info["concern"]]
        # Pretty-print the measure name
        nice_name = measure.replace("_", " ").title()
        classifications_md += f"- {emoji} **{nice_name}:** {info['label']}\n"

    # Highlight overall concern level
    overall_emoji = {
        "normal": "🟢",
        "warning": "🟡",
        "critical": "🔴"
    }[result["overall_concern"]]
    classifications_md += f"\n**Overall Concern:** {overall_emoji} {result['overall_concern'].upper()}"

    # ─────────────────────────────────────────────────────────
    # 4. Format the AI explanation
    # ─────────────────────────────────────────────────────────
    response_md = f"### 💬 Medical Explanation\n\n{result['response']}"

    # ─────────────────────────────────────────────────────────
    # 5. Format the sources used
    # ─────────────────────────────────────────────────────────
    sources_md = "### 📚 Sources Used\n\n"
    for src in result["sources"]:
        sources_md += f"- `{src}`\n"

    # ─────────────────────────────────────────────────────────
    # 6. Format the retrieved chunks (for transparency)
    # ─────────────────────────────────────────────────────────
    chunks_md = "### 🔍 Retrieved Evidence\n\n"
    for i, chunk in enumerate(result["retrieved_chunks"], 1):
        chunks_md += f"**Chunk #{i}** (rerank score: {chunk['rerank_score']:.2f})\n"
        chunks_md += f"*Source: {chunk['source']}*\n\n"
        chunks_md += f"> {chunk['text'][:300]}...\n\n---\n\n"

    return classifications_md, response_md, sources_md, chunks_md


# ─────────────────────────────────────────────────────────────
# Build the Gradio interface
# ─────────────────────────────────────────────────────────────
with gr.Blocks(title="IoT Health Monitor + Advanced RAG", theme=gr.themes.Soft()) as demo:

    # Header
    gr.Markdown("""
    # 🩺 IoT Health Monitor with Advanced RAG

    This system simulates IoT medical sensor readings and provides AI-powered
    medical explanations grounded in trusted clinical guidelines (WHO, ADA, JNC).

    **Pipeline:** IoT Sensors → Query Rewriting → Hybrid Search → Re-Ranking → SLM Generation
    """)

    # Two-column layout: inputs on left, outputs on right
    with gr.Row():
        # ─── LEFT COLUMN: Inputs ──────────────────────────────
        with gr.Column(scale=1):
            gr.Markdown("### 🩸 Sensor Readings")

            glucose_input = gr.Slider(
                minimum=40, maximum=500,
                value=120, step=1,
                label="Blood Glucose (mg/dL)",
                info="Normal: 70-99 | Hyperglycemia: >180"
            )

            bp_sys_input = gr.Slider(
                minimum=70, maximum=220,
                value=120, step=1,
                label="Blood Pressure - Systolic (mmHg)",
                info="Normal: <120 | Crisis: >180"
            )

            bp_dia_input = gr.Slider(
                minimum=40, maximum=140,
                value=80, step=1,
                label="Blood Pressure - Diastolic (mmHg)",
                info="Normal: <80 | Crisis: >120"
            )

            heart_rate_input = gr.Slider(
                minimum=30, maximum=180,
                value=72, step=1,
                label="Heart Rate (bpm)",
                info="Normal: 60-100 | Tachycardia: >100"
            )

            patient_age_input = gr.Slider(
                minimum=18, maximum=100,
                value=45, step=1,
                label="Patient Age (years)"
            )

            analyze_btn = gr.Button("🔍 Analyze Readings", variant="primary", size="lg")

            # Quick preset buttons for fast testing
            gr.Markdown("### ⚡ Quick Test Presets")
            with gr.Row():
                normal_btn = gr.Button("🟢 Normal Case", size="sm")
                warning_btn = gr.Button("🟡 Warning Case", size="sm")
                critical_btn = gr.Button("🔴 Critical Case", size="sm")

        # ─── RIGHT COLUMN: Outputs ────────────────────────────
        with gr.Column(scale=2):
            classifications_output = gr.Markdown(label="Classifications")
            response_output = gr.Markdown(label="Medical Explanation")

            # Collapsible details (for cleaner UI)
            with gr.Accordion("📚 Show Sources & Evidence", open=False):
                sources_output = gr.Markdown()
                chunks_output = gr.Markdown()

    # ─────────────────────────────────────────────────────────
    # Wire up the buttons to the analyze function
    # ─────────────────────────────────────────────────────────
    analyze_btn.click(
        fn=analyze,
        inputs=[glucose_input, bp_sys_input, bp_dia_input, heart_rate_input, patient_age_input],
        outputs=[classifications_output, response_output, sources_output, chunks_output]
    )

    # Preset buttons just update the slider values (don't auto-analyze)
    normal_btn.click(
        fn=lambda: (88, 117, 76, 78, 45),
        outputs=[glucose_input, bp_sys_input, bp_dia_input, heart_rate_input, patient_age_input]
    )
    warning_btn.click(
        fn=lambda: (165, 145, 92, 108, 55),
        outputs=[glucose_input, bp_sys_input, bp_dia_input, heart_rate_input, patient_age_input]
    )
    critical_btn.click(
        fn=lambda: (380, 195, 118, 145, 65),
        outputs=[glucose_input, bp_sys_input, bp_dia_input, heart_rate_input, patient_age_input]
    )


# ─────────────────────────────────────────────────────────────
# Launch the UI (share=True gives a public URL for testing)
# ─────────────────────────────────────────────────────────────
print("🚀 Launching Gradio UI...")
print("   Wait for the public URL to appear (takes ~10 seconds)")
demo.launch(share=True, debug=False)

/tmp/ipykernel_644/2858072334.py:90: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="IoT Health Monitor + Advanced RAG", theme=gr.themes.Soft()) as demo:


🚀 Launching Gradio UI...
   Wait for the public URL to appear (takes ~10 seconds)
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://51f407373126a02903.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [18]:
# ============================================================
# BLOCK 13: RAGAS Evaluation
# Measure RAG quality across multiple test cases
# ============================================================

# Install RAGAS-compatible packages if not already installed
!pip install -q ragas datasets langchain-huggingface 2>&1 | tail -3

import pandas as pd
from datasets import Dataset


# ─────────────────────────────────────────────────────────────
# 1. Generate evaluation test cases
# ─────────────────────────────────────────────────────────────
print("📊 Generating evaluation test cases...\n")

test_cases = []

# 2 cases per severity level for balanced evaluation
configs = [
    ("normal", 35), ("normal", 60),
    ("warning", 50), ("warning", 70),
    ("critical", 55), ("critical", 75),
]

for severity, age in configs:
    # Generate IoT reading
    reading = generate_iot_reading(severity=severity, patient_age=age)

    # Run full pipeline
    result = run_full_pipeline(reading)

    # Build a "ground truth" reference answer based on classifications
    # In a real eval, this would come from medical experts
    classifications_text = ", ".join([
        f"{m}: {info['label']}"
        for m, info in result["classifications"].items()
    ])
    ground_truth = (
        f"The readings indicate {classifications_text}. "
        f"Overall concern level is {result['overall_concern']}. "
        f"Patient should follow appropriate medical guidance."
    )

    test_cases.append({
        "question": result["query"],
        "answer": result["response"],
        "contexts": [c["text"] for c in result["retrieved_chunks"]],
        "ground_truth": ground_truth,
        "severity": severity
    })

    print(f"  ✓ Generated {severity} case (age {age})")

print(f"\n✓ Created {len(test_cases)} test cases")


# ─────────────────────────────────────────────────────────────
# 2. Convert to HuggingFace Dataset (RAGAS format)
# ─────────────────────────────────────────────────────────────
eval_data = {
    "question":     [tc["question"] for tc in test_cases],
    "answer":       [tc["answer"] for tc in test_cases],
    "contexts":     [tc["contexts"] for tc in test_cases],
    "ground_truth": [tc["ground_truth"] for tc in test_cases],
}
eval_dataset = Dataset.from_dict(eval_data)

print(f"\n📦 Dataset shape: {len(eval_dataset)} samples")


# ─────────────────────────────────────────────────────────────
# 3. Run RAGAS evaluation using local SLM as judge
# ─────────────────────────────────────────────────────────────
print("\n🔬 Running RAGAS evaluation...")
print("   (This takes 2-3 minutes — judge LLM evaluates each sample)\n")

from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_huggingface import HuggingFacePipeline, HuggingFaceEmbeddings
from transformers import pipeline as hf_pipeline

# Wrap our local Qwen2.5 SLM as the judge LLM for RAGAS
judge_pipeline = hf_pipeline(
    "text-generation",
    model=slm_model,
    tokenizer=tokenizer,
    max_new_tokens=300,
    temperature=0.1,        # very low for consistent judging
    do_sample=True,
    return_full_text=False
)
judge_llm = LangchainLLMWrapper(HuggingFacePipeline(pipeline=judge_pipeline))

# Use the same MiniLM embedder for any embedding needed by RAGAS
judge_embedder = LangchainEmbeddingsWrapper(
    HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
)

# Run evaluation across all 3 metrics
ragas_results = evaluate(
    dataset=eval_dataset,
    metrics=[faithfulness, answer_relevancy, context_precision],
    llm=judge_llm,
    embeddings=judge_embedder,
    raise_exceptions=False  # don't crash on individual sample errors
)

print("✓ Evaluation complete\n")


# ─────────────────────────────────────────────────────────────
# 4. Display results
# ─────────────────────────────────────────────────────────────
print("=" * 70)
print("📊 RAGAS EVALUATION RESULTS")
print("=" * 70)

# Convert to pandas for nicer display
results_df = ragas_results.to_pandas()

# Calculate aggregate scores
print("\n🎯 Overall Metrics (averaged across all test cases):")
print(f"   • Faithfulness:      {results_df['faithfulness'].mean():.3f}  (claims grounded in context)")
print(f"   • Answer Relevancy:  {results_df['answer_relevancy'].mean():.3f}  (answer addresses query)")
print(f"   • Context Precision: {results_df['context_precision'].mean():.3f}  (retrieved chunks useful)")

print("\n📋 Per-Case Breakdown:")
display_df = results_df[["faithfulness", "answer_relevancy", "context_precision"]].copy()
display_df.insert(0, "severity", [tc["severity"] for tc in test_cases])
display_df.index = [f"Case {i+1}" for i in range(len(display_df))]

# Round for readability
print(display_df.round(3).to_string())

print("\n" + "=" * 70)
print("✅ Evaluation complete — results ready for the paper")
print("=" * 70)

📊 Generating evaluation test cases...

  ✓ Generated normal case (age 35)
  ✓ Generated normal case (age 60)
  ✓ Generated warning case (age 50)
  ✓ Generated warning case (age 70)
  ✓ Generated critical case (age 55)
  ✓ Generated critical case (age 75)

✓ Created 6 test cases

📦 Dataset shape: 6 samples

🔬 Running RAGAS evaluation...
   (This takes 2-3 minutes — judge LLM evaluates each sample)



/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)
/tmp/ipykernel_644/3830472155.py:80: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, answer_relevancy, context_precision
/tmp/ipykernel_644/3830472155.py:80: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections imp

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/tmp/ipykernel_644/3830472155.py:99: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  judge_embedder = LangchainEmbeddingsWrapper(


Evaluating:   0%|          | 0/18 [00:00<?, ?it/s]

Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both

KeyboardInterrupt: 